# VoltSight - Modélisation (BPNN -> CNN)

Ce notebook relance toute la phase de modélisation.

Avant de commencer : menu *Exécution* -> *Modifier le type d'exécution* -> T4 GPU, puis *Exécution* -> Tout exécuter.

## 1. Récupérer le code et les données

In [ ]:
%cd /content
!rm -rf voltsight
!git clone -q https://github.com/Vamssy47/voltsight.git
%cd /content/voltsight
!git clone -q --depth 1 https://github.com/InsulatorData/InsulatorDataSet.git data/InsulatorDataSet
import torch; print('GPU disponible :', torch.cuda.is_available())

## 2. Vérifier la rétropropagation du BPNN
Une erreur d'environ 1e-8 prouve que les gradients sont correctement calculés.

In [ ]:
import sys, numpy as np
sys.path.insert(0, 'src')
from bpnn import BPNN
X = np.random.randn(30, 10); y = (np.random.rand(30) > 0.7).astype(float)
print('Erreur du gradient :', BPNN([10, 8, 5, 1], pos_weight=3.0).gradient_check(X, y))

## 3. Préparer les découpes d'isolateurs

In [ ]:
!python src/prepare_data.py
!python src/prepare_data.py --width 256 --height 64 --out data/cplid_crops_256.npz

## 4. Étape 2 - BPNN sur l'isolateur entier et test de masquage (environ 10 min)
Le BPNN obtient un bon score, mais le test de masquage montre qu'il ne regarde pas le défaut.

In [ ]:
!python -W ignore src/train_evaluate.py

## 5. Étape 3 - BPNN par zones (tâche corrigée)

In [ ]:
!python -W ignore src/patch_model.py
!python -W ignore src/patch_model.py --data data/cplid_crops_256.npz --patch 64 --out results/etape3_zones_64

## 6. Étapes 4 et 5 - CNN standard et contrefactuel, ensemble, règles de seuil (environ 5 min sur GPU)
Pour réutiliser les modèles déjà entraînés au lieu de réentraîner, ajouter `--reuse`.

In [ ]:
!python -W ignore src/cnn_model.py

## 7. Figures

In [ ]:
from IPython.display import Image, display
for f in ['results/etape2_isolateur_entier/roc_precision_rappel.png',
          'results/etape4_cnn/bpnn_vs_cnn.png',
          'results/etape4_cnn/cnn_contrefactuel_gradcam_masquage.png',
          'results/etape4_cnn/stabilite_ensemble.png']:
    print(f); display(Image(f))

## 8. Sauvegarder les résultats (Colab efface tout à la déconnexion)

In [ ]:
!zip -rq /content/resultats_voltsight.zip results
from google.colab import files
files.download('/content/resultats_voltsight.zip')